# AWS Glue 5.0 no Notisbook

Exemplo para o kernel Docker **AWS Glue 5.0**: `GlueContext`, `DynamicFrame`, arquivos na zona `landing` do Notislake e tabelas Iceberg e Delta externas na camada `bronze` do catálogo do app.

Execute as células em ordem. Os arquivos de exemplo ficam em `s3a://notislake/landing/glue_demo`.

In [ ]:
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session  # mesma sessão usada pela célula "spark sql" e pelo catálogo do app

# Zona landing do Notislake (externa ao catálogo): arquivos soltos, sem tabela registrada.
LANDING = "s3a://notislake/landing/glue_demo"
print(LANDING)

## Arquivos CSV, JSON e Parquet

Geramos um DataFrame pequeno e gravamos nos três formatos na zona `landing`.

In [ ]:
df = spark.createDataFrame(
    [(1, "Ana", "Dados"), (2, "Bruno", "Plataforma"), (3, "Carla", "Segurança")],
    ["id", "nome", "area"],
)

df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{LANDING}/csv")
df.coalesce(1).write.mode("overwrite").json(f"{LANDING}/json")
df.coalesce(1).write.mode("overwrite").parquet(f"{LANDING}/parquet")

## Leitura com `create_dynamic_frame.from_options`

O Glue usa `connection_type="s3"`; aqui lemos de volta os arquivos gravados na zona `landing`.

In [ ]:
dyf_csv = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [f"{LANDING}/csv"]},
    format="csv",
    format_options={"withHeader": True},
)
dyf_json = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [f"{LANDING}/json"]},
    format="json",
)
dyf_parquet = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [f"{LANDING}/parquet"]},
    format="parquet",
)

for nome, dyf in [("csv", dyf_csv), ("json", dyf_json), ("parquet", dyf_parquet)]:
    print(nome, dyf.count())
dyf_parquet.printSchema()

## Gravação com `write_dynamic_frame.from_options`

O resultado também fica na zona `landing`. O destino é apagado antes da gravação (o write do Glue só acrescenta arquivos), então cada execução produz o mesmo resultado.

In [ ]:
destino = spark._jvm.org.apache.hadoop.fs.Path(f"{LANDING}/saida")
destino.getFileSystem(sc._jsc.hadoopConfiguration()).delete(destino, True)

glueContext.write_dynamic_frame.from_options(
    frame=dyf_parquet,
    connection_type="s3",
    connection_options={"path": f"{LANDING}/saida"},
    format="parquet",
)
spark.read.parquet(f"{LANDING}/saida").show()

## `DynamicFrame` e `DataFrame`

`toDF()` converte para DataFrame e `DynamicFrame.fromDF` faz o caminho inverso.

In [ ]:
from pyspark.sql.functions import upper

df2 = dyf_parquet.toDF().withColumn("nome_maiusculo", upper("nome"))
dyf2 = DynamicFrame.fromDF(df2, glueContext, "dyf2")
dyf2.toDF().show()

## Tabela Iceberg externa no catálogo do app

O catálogo `iceberg` é servido pelo app; a tabela fica na camada `bronze`, com arquivos em `s3a://notislake/bronze/`, e aparece no painel Catalog. `createOrReplace` permite reexecutar a célula.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.bronze")

(
    dyf2.toDF().writeTo("iceberg.bronze.glue_t")
    .tableProperty("location", "s3a://notislake/bronze/glue_t_files")
    .createOrReplace()
)

spark.table("iceberg.bronze.glue_t").show()

## Tabela Delta externa no `spark_catalog`

`DeltaTable` está disponível no kernel Docker. A tabela é externa: o `path` fica em `s3a://notislake/bronze/` e `overwrite` permite reexecutar a célula.

In [ ]:
from delta.tables import DeltaTable

(
    df.write.format("delta")
    .option("path", "s3a://notislake/bronze/glue_delta_t_files")
    .mode("overwrite")
    .saveAsTable("bronze.glue_delta_t")
)
DeltaTable.forName(spark, "bronze.glue_delta_t").history().select("version", "operation").show()

## Célula "spark sql"

A célula do tipo **spark sql** do app executa na mesma sessão (`spark`) criada acima, então enxerga as tabelas deste notebook. Exemplo de conteúdo:

```sql
SELECT * FROM iceberg.bronze.glue_t
```

## Limitações

- `getResolvedOptions(sys.argv, ['JOB_NAME'])` não funciona no notebook, pois não há argumentos de job. Defina os valores direto numa variável.
- `glueContext.create_dynamic_frame.from_catalog` (Glue Data Catalog remoto) não é suportado: sem AWS a chamada falha rápido. Use o catálogo do app (`iceberg` e `spark_catalog`).
- Para tabelas Iceberg externas o app recusa o `PURGE`: use `DROP TABLE` simples para desregistrar.